# Deep Agents: Building a Travel Planner from Scratch

In this notebook we'll build a **travel planning agent** progressively, layering one Deep Agents capability at a time. By the end we'll have an orchestrator that delegates to three specialist subagents (hotels, flights, activities), uses long-term memory to remember a traveler's preferences across sessions, gates real-money bookings behind human approval, and produces polished itineraries through reusable skills.

**What we'll cover, in order:**

<table style="font-size: 1.15em">
<tr><th>Part</th><th>Concept</th></tr>
<tr><td>0</td><td>Setup &amp; installation</td></tr>
<tr><td>1</td><td>Your first deep agent (the harness)</td></tr>
<tr><td>2</td><td>Adding custom tools (Tavily + mock pricing)</td></tr>
<tr><td>3</td><td>Backends: ephemeral, disk, persistent store, composite routing</td></tr>
<tr><td>4</td><td>Subagents: hotel-search, flight-search, activity-search</td></tr>
<tr><td>5</td><td>Middleware: built-ins + custom tool-call logging</td></tr>
<tr><td>6</td><td>Human-in-the-loop: approval before booking</td></tr>
<tr><td>7</td><td>Long-term memory: semantic, episodic, procedural + per-user namespacing</td></tr>
<tr><td>8</td><td>AGENTS.md &amp; Skills (itinerary, budget, packing list, travel brief)</td></tr>
<tr><td>9</td><td>The complete travel planner</td></tr>
<tr><td>10</td><td>Deploying with the <code>deepagents</code> CLI</td></tr>
<tr><td>11</td><td>Next steps</td></tr>
</table>


In [1]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* Scale the rendered markdown container -- headings, paragraphs, lists, tables all
   use relative (em) sizing internally, so a single rule on the container scales
   everything proportionally. Targets both JupyterLab/Notebook 7 and classic Jupyter. */
.jp-RenderedMarkdown,
.text_cell_render,
div.markdown {
    font-size: 1.2em;
    line-height: 1.55;
}

/* Inline code in markdown -- drop the grey background, render bold green instead.
   The :not(pre) > code selector means "code that's NOT inside a pre block",
   so multi-line ``` fenced code blocks keep their normal styling. */
.jp-RenderedMarkdown :not(pre) > code,
.text_cell_render :not(pre) > code {
    background-color: transparent;
    color: #16a34a;          /* readable green in both light and dark themes */
    font-weight: 700;
    padding: 0;
    border: none;
    font-size: 0.95em;       /* slightly smaller so bolding doesn't bloat the line */
}
</style>
"""))


## Part 0: Setup & Installation

Install the dependencies and initialize a model. We use OpenAI's `gpt-4.1-mini` for speed; any tool-calling chat model works.


In [119]:
# Install required packages
# Run uv sync to install the packages, or:
# !pip install deepagents tavily-python python-dotenv langchain langgraph


### Initialize your LLM

`init_chat_model` gives you a single uniform interface across providers (OpenAI, Anthropic, Bedrock, Azure). Swap the string and the rest of the notebook still works.


In [3]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env", override=True)

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-4.1-mini")

import warnings
warnings.filterwarnings('ignore', message='LangSmith now uses UUID v7')


## Part 1: Your First Deep Agent (The Harness)

`create_deep_agent` returns a LangGraph agent pre-wired with three middleware capabilities you'll use in every Deep Agent:

1. **TodoListMiddleware** — gives the agent a `write_todos` tool for planning
2. **FilesystemMiddleware** — gives it `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
3. **SubAgentMiddleware** — gives it a `task()` tool to delegate to subagents (Part 4)

Plus summarization and prompt-cache middleware that keep long conversations affordable.

Here's the smallest possible travel agent:


In [4]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a helpful travel planning assistant. "
        "When referencing file paths, use backtick formatting like `path/file.md` "
        "instead of markdown links."
    ),
)


### Test the built-in filesystem tools

The agent already has a virtual filesystem. Let's ask it to brainstorm trip ideas and save them.


In [5]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Brainstorm 3 long-weekend trip ideas from San Francisco for someone "
            "who loves food and architecture. Save the list to `/trip_ideas.md`, "
            "then list the files you have."
        ),
    }]
})

print(result["messages"][-1].content)


I have saved the 3 long-weekend trip ideas for a food and architecture lover from San Francisco to the file /trip_ideas.md. Currently, the only file in the directory is:

- /trip_ideas.md

Let me know if you want to see the contents or need any other help.


In [6]:
# THE VIRTUAL FILESYSTEM LIVES IN result["files"]

for path, file_data in result.get("files", {}).items():
    print(f"=== {path} ===")
    print("\n".join(file_data["content"]))
    print()


=== /trip_ideas.md ===
# Long-Weekend Trip Ideas from San Francisco for Food and Architecture Lovers

1. Napa Valley, CA
- Famous for its vineyards and wine tasting experiences.
- Explore iconic winery estates with beautiful architectural designs.
- Enjoy gourmet farm-to-table dining at acclaimed restaurants.

2. Santa Barbara, CA
- Known as the American Riviera with stunning Spanish Colonial Revival architecture.
- Visit the historic Santa Barbara County Courthouse and Old Mission.
- Experience fresh seafood and vibrant food scene inspired by local ingredients.

3. Portland, OR
- Renowned for its innovative food culture with many food trucks and farm-to-table eateries.
- Architectural variety from historic neighborhoods to modern eco-buildings.
- Explore landmarks like the Portland Art Museum and Pittock Mansion.




### Key Takeaway

A Deep Agent isn't just an LLM — it's a **harness** of middleware that gives the model a planner, a virtual filesystem, and a delegation primitive out of the box. You haven't written any tools yet, and the agent can already plan, write files, and read them back.


## Part 2: Adding Custom Tools

Travel planning needs both **discovery** (what hotels exist?) and **details** (what does it cost?). We'll define two flavors of tools:

- **Tavily-backed discovery** — one `search_travel(query, category)` tool that injects the right `site:` filters per category. Real web results.
- **Mock pricing tools** — deterministic Python functions that return fake quotes/rates/availability. They make the rest of the notebook reproducible without burning API quota and give us bookable IDs for the human-in-the-loop section later.

Subagents in Part 4 will each get the relevant subset of these tools.


In [73]:
from typing import Literal
from langchain_core.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

# --- DISCOVERY TOOL (real web search, scoped by category) ---

CATEGORY_SITES = {
    "hotel": "site:booking.com OR site:hotels.com OR site:tripadvisor.com",
    "flight": "site:kayak.com OR site:google.com/flights OR site:skyscanner.com",
    "activity": "site:tripadvisor.com OR site:viator.com OR site:getyourguide.com",
}

@tool(parse_docstring=True)
def search_travel(query: str, category: Literal["hotel", "flight", "activity"]) -> str:
    """Search the web for travel info, scoped to a category.

    Args:
        query: Free-text search (e.g. "boutique hotels Lisbon Alfama October").
        category: One of "hotel", "flight", or "activity". Determines which sites are searched.
    """
    scoped = f"{query} {CATEGORY_SITES[category]}"
    results = tavily_client.search(scoped, max_results=3, topic="general")
    out = []
    for r in results.get("results", []):
        out.append(f"- {r['title']}\n  {r['url']}\n  {r['content'][:300]}")
    return "\n\n".join(out) if out else "No results."

# ---   MOCK PRICING/AVAILABILITY TOOLS (deterministic) ---

@tool(parse_docstring=True)
def get_flight_quotes(origin: str, destination: str, date: str) -> str:
    """Get mock flight quotes between two cities on a date.

    Args:
        origin: Origin city or IATA code.
        destination: Destination city or IATA code.
        date: Departure date in YYYY-MM-DD.
    """
    base = (hash((origin, destination, date)) % 400) + 250
    return "\n".join([
        f"FL-{base}A | United UA{100+base%900} | depart 08:15 arrive 11:40 | nonstop | ${base}",
        f"FL-{base}B | Delta DL{100+(base+13)%900} | depart 13:05 arrive 17:30 | nonstop | ${base+45}",
        f"FL-{base}C | American AA{100+(base+27)%900} | depart 21:50 arrive 05:10+1 | 1 stop | ${base-60}",
    ])

@tool(parse_docstring=True)
def get_hotel_rates(city: str, checkin: str, checkout: str) -> str:
    """Get mock nightly hotel rates for a city and date range.

    Args:
        city: Destination city.
        checkin: Check-in date YYYY-MM-DD.
        checkout: Check-out date YYYY-MM-DD.
    """
    base = (hash((city, checkin)) % 200) + 120
    return "\n".join([
        f"HT-{base}A | The Garden Boutique ({city}) | 4.6star | ${base}/night | refundable",
        f"HT-{base}B | Grand Plaza ({city}) | 4.3star | ${base+90}/night | non-refundable",
        f"HT-{base}C | Riverside Inn ({city}) | 4.8star | ${base+40}/night | refundable",
    ])

@tool(parse_docstring=True)
def get_activity_options(city: str, date: str) -> str:
    """Get mock activity / tour options for a city on a date.

    Args:
        city: Destination city.
        date: Date YYYY-MM-DD.
    """
    base = (hash((city, date)) % 60) + 25
    return "\n".join([
        f"AC-{base}A | Half-day food walking tour | 3h | small group | ${base}",
        f"AC-{base}B | Old-town architecture tour | 2h | private guide | ${base+35}",
        f"AC-{base}C | Sunset harbor cruise | 2h | drinks included | ${base+20}",
    ])


### Add custom tools to our agent

Recreate the agent with all four travel tools. We're not using subagents yet — the orchestrator can call everything directly so you can see how a single agent uses tools before we split it up in Part 4.


In [74]:
agent = create_deep_agent(
    model=model,
    tools=[search_travel, get_flight_quotes, get_hotel_rates, get_activity_options],
    system_prompt=(
        "You are a travel planning assistant. "
        " Use search_travel for discovery "
        "and the get_* tools for prices and availability. Save findings to files. "
        "When referencing file paths, use backtick formatting like `path/file.md`."
    ),
)


In [75]:
# QUICK SMOKE TEST

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "I'm thinking about a 4-day trip to Lisbon in late September. "
            "Pull a few hotel rates and flight quotes from SFO so I can compare."
        ),
    }]
})

print(result["messages"][-1].content)


Here are some flight options from SFO to Lisbon on September 25, 2024:
- United UA735, depart 08:15 arrive 11:40, nonstop, $635
- Delta DL748, depart 13:05 arrive 17:30, nonstop, $680
- American AA762, depart 21:50 arrive 05:10+1, 1 stop, $575

And some hotel options in Lisbon for a 4-night stay from September 25 to 29, 2024:
- The Garden Boutique, 4.6 stars, $221/night, refundable
- Grand Plaza, 4.3 stars, $311/night, non-refundable
- Riverside Inn, 4.8 stars, $261/night, refundable

Let me know if you want more options or details!


### Key Takeaway

Custom tools plug into the same agent the same way they would in any LangChain agent — `@tool` and pass them in via `tools=`. The Deep Agents harness adds its built-in tools alongside yours.


## Part 3: Understanding Backends

The virtual filesystem the agent has been writing to is backed by a **Backend**. Deep Agents ship four:

<table style="font-size: 1.35em">
<tr><th>Backend</th><th>Storage</th><th>Persistence</th></tr>
<tr><td><code>StateBackend</code></td><td>LangGraph state</td><td>Per-thread (default)</td></tr>
<tr><td><code>FilesystemBackend</code></td><td>Real disk</td><td>Permanent, on the host</td></tr>
<tr><td><code>StoreBackend</code></td><td>LangGraph <code>BaseStore</code></td><td>Permanent, cross-thread (great for memory)</td></tr>
<tr><td><code>CompositeBackend</code></td><td>Routes paths between the above</td><td>&mdash;</td></tr>
</table>

A **checkpointer** lets state survive across `invoke()` calls within the same thread. We'll use `MemorySaver` here; in production you'd swap in `MongoDB` or `Postgres` as a checkpointer.


In [76]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[search_travel, get_flight_quotes, get_hotel_rates, get_activity_options],
    system_prompt="You are a travel planning assistant.",
    checkpointer=checkpointer,
)

config_a = {"configurable": {"thread_id": str(uuid7())}}    # THREAD_ID IS HOW WE GROUP MESSAGES INTO BACK-AND-FORTH CONVERSATIONS


In [77]:
# WRITE A FILE IN THIS THREAD

agent.invoke(
    {"messages": [{"role": "user", "content": "Save 'Lisbon, Sep 25-29, food + architecture focus' to `/trip_brief.md`."}]},
    config=config_a,
)
files = agent.get_state(config_a).values.get("files", {})
print("Files in thread A:", list(files))


Files in thread A: ['/trip_brief.md']


In [78]:
# IN THE SAME THREAD, THE FILE PERSISTS ACROSS INVOCATIONS

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Read `/trip_brief.md` and tell me what's in it."}]},
    config=config_a,
)
print(result["messages"][-1].content)


The file /trip_brief.md contains: "Lisbon, Sep 25-29, food + architecture focus".


In [79]:
# IN A NEW THREAD, THE FILE IS GONE -- StateBackend IS PER-THREAD

config_b = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "List the files you have."}]},
    config=config_b,
)
print(result["messages"][-1].content)


There are no files available in the root directory. Let me know if you want to create or upload files, or if you want to proceed with any other request.


### FilesystemBackend - Writing to Real Disk

Often you actually want files on disk (so a developer can inspect them, or another process can read them). `FilesystemBackend` writes to a real directory.


In [82]:
from deepagents.backends import FilesystemBackend
import tempfile, os

trip_dir = tempfile.mkdtemp(prefix="trip_")
print(f"Files will land in: {trip_dir}")

disk_agent = create_deep_agent(
    model=model,
    tools=[search_travel, get_flight_quotes, get_hotel_rates, get_activity_options],
    system_prompt="You are a travel planning assistant.",
    backend=lambda rt: FilesystemBackend(root_dir=trip_dir, virtual_mode=True),
    checkpointer=MemorySaver(),
)


Files will land in: /var/folders/qz/wzwyh67n3w11r2k29h5rp49c0000gn/T/trip_45rfnty2


In [83]:
# TEST: ASK IT TO SAVE A DRAFT ITINERARY, THEN PEEK AT THE DISK

disk_agent.invoke(
    {"messages": [{"role": "user", "content": "Write a one-line draft itinerary for a Lisbon trip to `/itinerary.md`."}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("On disk:", os.listdir(trip_dir))
print("Contents:", Path(trip_dir, "itinerary.md").read_text())


On disk: ['itinerary.md']
Contents: Day 1: Explore Alfama district, visit São Jorge Castle, and enjoy a traditional Fado evening.
Day 2: Discover Belém with Jerónimos Monastery, Belém Tower, and try pastéis de nata.
Day 3: Day trip to Sintra to see Pena Palace and Quinta da Regaleira, return to Lisbon for dinner.
Day 4: Visit Lisbon Oceanarium, stroll through Parque das Nações, and shop at Avenida da Liberdade.
Day 5: Relax in the LX Factory, explore the Bairro Alto nightlife.



### CompositeBackend - Routing Paths to Different Backends

The most powerful backend. Route some paths to ephemeral state, others to disk, others to persistent storage. We'll use this in Part 7 to keep scratch files in state but route `/memories/` to a persistent store.


In [84]:
from deepagents.backends import StateBackend, CompositeBackend

def composite_factory(rt):
    return CompositeBackend(
        default=StateBackend(rt),
        routes={
            "/notes/": FilesystemBackend(root_dir=trip_dir, virtual_mode=True),
        },
    )

routing_agent = create_deep_agent(
    model=model,
    tools=[search_travel, get_flight_quotes, get_hotel_rates, get_activity_options],
    system_prompt="You are a travel planning assistant.",
    backend=composite_factory,
    checkpointer=MemorySaver(),
)

In [85]:
# FILES IN /NOTES/ GO TO DISK; EVERYTHING ELSE STAYS IN STATE

routing_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Write 'scratch idea' to `/scratch.md` and 'persistent note' to `/notes/keep.md`."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)

print("On disk under /notes/:", os.listdir(trip_dir))


On disk under /notes/: ['itinerary.md', 'keep.md']


### StoreBackend - Cross-Thread Persistence

`StoreBackend` is the right choice for **long-term memory** — files written in one conversation should be readable in another. We'll set this up properly in Part 7 once we've talked about subagents.

**Long-term memory** is what lets your agent remember your users across multiple sessions.


In [87]:
# Cleanup the temp directory
import shutil
shutil.rmtree(trip_dir, ignore_errors=True)

### Key Takeaway

Backends are pluggable. The same agent code can write to ephemeral state (great for dev/testing), real disk (great for inspection), or a persistent store (great for long-term memory). `CompositeBackend` lets you mix-and-match: scratch in state, memories in store.


## Part 4: Adding Subagents (Hotel, Flight, Activity)

A single agent juggling 7+ tools and a multi-step trip can lose focus. Subagents fix this by giving each specialist its own:

- **Isolated context window** — the orchestrator doesn't see the subagent's tool outputs, only its final report
- **Smaller toolset** — easier for the model to choose
- **Focused system prompt** — narrowly scoped instructions

We'll define **three** subagents — `hotel-search`, `flight-search`, `activity-search` — and an orchestrator that delegates to all three (often in parallel) and synthesizes the results into an itinerary.


In [88]:
from datetime import datetime
from deepagents import SubAgent

today = datetime.now().strftime("%Y-%m-%d")

hotel_subagent: SubAgent = {
    "name": "hotel-search",
    "description": (
        "Find lodging options for a city and date range. "
        "Provide city, check-in, check-out, and any preferences (budget, neighborhood, amenities)."
    ),
    "system_prompt": (
        f"You are a hotel research specialist. Today is {today}.\n\n"
        "1. Use `search_travel(category='hotel')` to discover options (max 2 calls).\n"
        "2. Use `get_hotel_rates` for concrete pricing.\n"
        "3. Return a short markdown list: 3-5 options with hotel ID, name, price/night, and a one-line note.\n"
        "Do NOT recommend a single winner -- return options."
    ),
    "tools": [search_travel, get_hotel_rates],
}

flight_subagent: SubAgent = {
    "name": "flight-search",
    "description": (
        "Find flight options between two cities for a given date. "
        "Provide origin, destination, and date."
    ),
    "system_prompt": (
        f"You are a flight research specialist. Today is {today}.\n\n"
        "1. Optionally use `search_travel(category='flight')` for context (max 1 call).\n"
        "2. Use `get_flight_quotes` for concrete itineraries.\n"
        "3. Return a markdown list: 3 options with flight ID, airline, times, stops, price."
    ),
    "tools": [search_travel, get_flight_quotes],
}

activity_subagent: SubAgent = {
    "name": "activity-search",
    "description": (
        "Find activities, tours, and attractions for a city on a date. "
        "Provide city, date, and any interests (food, history, nightlife, outdoors)."
    ),
    "system_prompt": (
        f"You are a local activities specialist. Today is {today}.\n\n"
        "1. Use `search_travel(category='activity')` to discover what's notable (max 2 calls).\n"
        "2. Use `get_activity_options` for bookable tours and prices.\n"
        "3. Return a markdown list: 4-6 activities with ID, name, duration, price, and a one-line description."
    ),
    "tools": [search_travel, get_activity_options],
}


In [89]:
ORCHESTRATOR_PROMPT = """You are a travel planning coordinator.

When asked to plan a trip:
1. Use `write_todos` to outline your plan.
2. Delegate research to the specialist subagents using the `task()` tool, IN PARALLEL where possible:
   - `hotel-search` for lodging
   - `flight-search` for flights
   - `activity-search` for things to do
3. Synthesize the subagent reports into a single itinerary saved to `/itinerary.md`.

When referencing file paths, use backtick formatting like `path/file.md`.
"""

orchestrator = create_deep_agent(
    model=model,
    system_prompt=ORCHESTRATOR_PROMPT,
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    checkpointer=MemorySaver(),
)

result = orchestrator.invoke(
    {"messages": [{"role": "user", "content": (
        "Plan 5 days in Tokyo, late June, mid-budget, food-focused. I'll fly from SFO."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print(result["messages"][-1].content)
print("\n\n")

file_data = result["files"]["/itinerary.md"]
content = file_data["content"]
print("Contents:", "\n".join(content) if isinstance(content, list) else content)


from langchain_core.messages import AIMessage, ToolMessage

# NOW LET'S PRINT OUT ORCHESTRATOR'S TODO LIST

print("\n--- Planner todos ---")
for msg in result["messages"]:
    if not isinstance(msg, AIMessage):
          continue
    for tc in msg.tool_calls or []:
          if tc["name"] != "write_todos":
              continue
          todos = tc["args"].get("todos", [])
          print(f"\n[write_todos] ({len(todos)} items)")
          for i, todo in enumerate(todos, 1):
              # todos are dicts like {"content": "...", "status":"pending"|"in_progress"|"completed"}
              status = todo.get("status", "?")
              content = todo.get("content", todo)
              marker = {"completed": "✓", "in_progress": "▶", "pending": "·"}.get(status, "·")
              print(f"  {marker} {i}. {content}")

# # NOW LET'S PRINT OUT THE SUBAGENT CALLS TO VERIFY THEY WERE CALLED
# print("\n--- Subagent calls ---")
# for msg in result["messages"]:
#     if isinstance(msg, AIMessage):
#           for tc in msg.tool_calls or []:
#               if tc["name"] == "task":
#                   args = tc["args"]
#                   subagent = args.get("subagent_type", "?")
#                   description = args.get("description", "")
#                   print(f"\n→ task({subagent}): {description}")


I have created a detailed 5-day Tokyo itinerary for late June, mid-budget, and food-focused. It includes flight options from SFO, hotel choices, daily food tours, local dining, and market visits.

I saved the itinerary to `/itinerary.md`.

Would you like help booking flights or hotels? Or would you like me to find notable famous sushi or ramen restaurants for reservations?



Contents: # 5-Day Tokyo Itinerary - Late June, Mid-Budget, Food-Focused

## Flights (from SFO to Tokyo)
- United Airlines (UA570): Non-stop, Departure 08:15, Arrival 11:40, Price: $470
- Delta Airlines (DL583): Non-stop, Departure 13:05, Arrival 17:30, Price: $515
- American Airlines (AA597): 1 stop, Departure 21:50, Arrival 05:10 (+1 day), Price: $410

---

## Hotel Options (5 nights)
Choose one based on preference:
- The Garden Boutique: $236/night, highly rated, refundable
- Riverside Inn: $276/night, comfortable, refundable
- Grand Plaza: $326/night, good amenities, non-refundable

---

## Day 1: Arrival & Loc

**NOTE:** Each subagent runs in an isolated context. The orchestrator only sees the *final report* each subagent returns -- not the dozen intermediate tool calls. That's how you scale Deep Agents to long, complex tasks without blowing the context window.


### Key Takeaway

Subagents are isolated, specialized agents you delegate to via the `task()` tool. They have their own tools, prompts, and context. The orchestrator stays high-level: plan, delegate, synthesize, save.


## Part 5: Middleware Deep Dive

Every Deep Agent is built from middleware. The defaults we've been using:

<table style="font-size: 1.15em">
<tr><th>Middleware</th><th>What it adds</th></tr>
<tr><td><code>TodoListMiddleware</code></td><td><code>write_todos</code> tool for planning</td></tr>
<tr><td><code>FilesystemMiddleware</code></td><td><code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>, <code>execute</code></td></tr>
<tr><td><code>SubAgentMiddleware</code></td><td><code>task()</code> tool for delegation</td></tr>
<tr><td><code>SummarizationMiddleware</code></td><td>Auto-compacts conversation when it gets long</td></tr>
<tr><td><code>AnthropicPromptCachingMiddleware</code></td><td>Cache prefixes for Anthropic models</td></tr>
</table>

You can add your own. Let's see todos in action first, then build a custom **tool-call logger** so we can watch the orchestrator + subagents fanning out.


In [90]:
# RUN A SMALL TASK AND LOOK AT THE TODOS THE PLANNER MIDDLEWARE PRODUCED

todo_run = orchestrator.invoke(
    {"messages": [{"role": "user", "content": (
        "Plan 2 days in Mexico City in November from SFO. "
        "Just outline the plan; you don't need to delegate yet."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)

for t in todo_run.get("todos", []):
    print(f"  [{t.get('status', '?'):>11}] {t.get('content', '')}")


  [in_progress] Outline the plan for a 2-day trip to Mexico City from SFO in November
  [    pending] Search for flights from SFO to Mexico City for the specified dates
  [    pending] Search for hotel options in Mexico City for 2 nights in November
  [    pending] Research activities and attractions to do in Mexico City for 2 days
  [    pending] Synthesize flight, hotel, and activity options into a full itinerary


### Custom Middleware: Logging Tool Calls

`@wrap_tool_call` lets you intercept every tool invocation. We'll log the tool name and a short preview of the args, then call `handler(request)` to run the tool unchanged. This is invaluable for understanding what your agent is actually doing -- especially with parallel subagents.


In [91]:
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def log_tool_calls(request, handler):
    """Print every tool call before and after execution."""
    name = request.tool_call["name"]
    args = request.tool_call.get("args", {})
    # For task() calls, surface which subagent is being invoked
    if name == "task":
        sub = args.get("subagent_type", "?")
        desc = (args.get("description") or "")[:80]
        print(f"\u27a4  task -> {sub}: {desc}...")
    else:
        preview = ", ".join(f"{k}={str(v)[:40]}" for k, v in args.items())
        print(f"\u27a4  {name}({preview})")
    result = handler(request)
    print(f"\u2713  {name} done")
    return result


In [92]:
# RECREATE THE ORCHESTRATOR WITH OUR LOGGING MIDDLEWARE APPENDED

logged_orchestrator = create_deep_agent(
    model=model,
    system_prompt=ORCHESTRATOR_PROMPT,
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    middleware=[log_tool_calls],
    checkpointer=MemorySaver(),
)

result = logged_orchestrator.invoke(
    {"messages": [{"role": "user", "content": (
        "Plan a quick 3-day trip to Barcelona in mid-October from SFO. "
        "I want a couple of hotel and flight options plus one activity per day."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("\n--- FINAL RESPONSE ---")
print(result["messages"][-1].content)


➤  write_todos(todos=[{'content': 'Research flight options fr)
✓  write_todos done
➤  task -> hotel-search: Find hotel options in Barcelona for a 3-night stay in mid-October. Provide a cou...
➤  task -> activity-search: Find one interesting activity for each of 3 days in Barcelona in mid-October. In...
➤  task -> flight-search: Find flight options from San Francisco (SFO) to Barcelona (BCN) for mid-October....
✓  task done
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Research flight options fr)
✓  write_todos done
➤  write_file(file_path=/itinerary.md, content=# 3-Day Trip to Barcelona from San Franc)
✓  write_file done

--- FINAL RESPONSE ---
I have planned your 3-day trip to Barcelona from San Francisco in mid-October. The itinerary with flight options, hotel choices, and one activity per day is saved in `/itinerary.md`.

Summary:
- 3 different flight options including nonstop and 1-stop flights.
- 3 hotel options with different rates and policies.
- Day 1: Food Walki

### Key Takeaway

Middleware is the extension point of Deep Agents.

The built-ins give you planning, files, and delegation; `@wrap_tool_call` lets you bolt on observability, retries, caching, or arg-rewriting in a few lines. The same hook works for orchestrator and subagents alike — you'll see `task()` calls fan out as each subagent is called.


## Part 6: Human-in-the-Loop

Some tool calls shouldn't run without your blessing — for a travel agent, **bookings**. Deep Agents ships HITL via `interrupt_on={tool_name: True}`. When the agent tries to call a gated tool, the graph pauses, the call surfaces in the result as `__interrupt__`, and you resume with `Command(resume=...)` after deciding `approve` / `reject` / `edit`.

Let's add bookable mock tools and gate them.


You can also customize *which* decisions are allowed per tool via `InterruptOnConfig` (`approve`/`edit`/`reject`/`response`) — useful if you want to allow editing a quote before approving it. We'll use the simplest form (`True` = always pause, all decisions allowed) here.


In [94]:
@tool(parse_docstring=True)
def book_flight(flight_id: str) -> str:
    """Book a flight by its ID. PRETEND this charges a credit card.

    Args:
        flight_id: A flight ID like 'FL-321A' returned by get_flight_quotes.
    """
    return f"BOOKED {flight_id} -- confirmation #CONF-{abs(hash(flight_id)) % 100000:05d}"

@tool(parse_docstring=True)
def book_hotel(hotel_id: str) -> str:
    """Book a hotel by its ID. PRETEND this charges a credit card.

    Args:
        hotel_id: A hotel ID like 'HT-184B' returned by get_hotel_rates.
    """
    return f"BOOKED {hotel_id} -- confirmation #CONF-{abs(hash(hotel_id)) % 100000:05d}"

hitl_checkpointer = MemorySaver()

agent_with_hitl = create_deep_agent(
    model=model,
    tools=[get_flight_quotes, get_hotel_rates, book_flight, book_hotel],
    system_prompt=(
        "You are a travel concierge. You may research and quote freely, "
        "but call book_flight / book_hotel only when the user explicitly asks to book."
    ),
    interrupt_on={
        "book_flight": True,
        "book_hotel": True,
    },
    checkpointer=hitl_checkpointer,
)


#### Let's give it a try

Ask the agent to find a flight and book the cheapest. It will pause before the booking call.


In [95]:
from langgraph.types import Command
import json

hitl_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_hitl.invoke(
    {"messages": [{"role": "user", "content": (
        "Find me a flight from SFO to LIS on 2026-09-23, then BOOK the cheapest one."
    )}]},
    config=hitl_config,
)

if result.get("__interrupt__"):
    interrupt = result["__interrupt__"][0]
    print("=== AGENT PAUSED FOR APPROVAL ===")
    print(json.dumps(interrupt.value, indent=2, default=str))
else:
    print("No interrupt -- agent didn't try to book.")
    print(result["messages"][-1].content)


=== AGENT PAUSED FOR APPROVAL ===
{
  "action_requests": [
    {
      "name": "book_flight",
      "args": {
        "flight_id": "FL-526C"
      },
      "description": "Tool execution requires approval\n\nTool: book_flight\nArgs: {'flight_id': 'FL-526C'}"
    }
  ],
  "review_configs": [
    {
      "action_name": "book_flight",
      "allowed_decisions": [
        "approve",
        "edit",
        "reject"
      ]
    }
  ]
}


#### Resume with approval

Pass `Command(resume={"decisions": [{"type": "approve"}]})` to continue. Use `"reject"` to refuse.


In [96]:
if result.get("__interrupt__"):
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=hitl_config,
    )
    print(result["messages"][-1].content)


I found flights from SFO to LIS on 2026-09-23. The cheapest flight is with American Airlines (AA653), departing at 21:50 with 1 stop, priced at $466. I have booked this flight for you. The confirmation number is CONF-62610.


### Key Takeaway

`interrupt_on` lets you put a **human gate** in front of any tool. Combined with the checkpointer, the agent's full state survives the pause — you can resume hours later, on a different machine, after approval lands in a Slack message or a PR review. Booking, sending email, deleting data: all good candidates.


## Part 7: Long-Term Memory

`StateBackend` is per-thread. For an agent that should remember **across conversations** — the traveler's seat preference, dietary needs, loyalty numbers — we need the `StoreBackend`, which writes to LangGraph's persistent `BaseStore`.

Pattern: route `/memories/` to a `StoreBackend`, leave everything else in `StateBackend`. The agent uses the same `read_file`/`write_file` tools either way; the backend handles persistence.


In [97]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

memory_store = InMemoryStore()

def memory_backend(rt):
    return CompositeBackend(
        default=StateBackend(rt),
        routes={
            "/memories/": StoreBackend(rt, namespace=lambda ctx: ("memories",)),
        },
    )

In [98]:
memory_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a travel concierge with long-term memory. "
        "Save lasting facts about the traveler to `/memories/preferences.md`. "
        "Read it at the start of each conversation."
    ),
    backend=memory_backend,
    checkpointer=MemorySaver(),
    store=memory_store,
)

In [99]:
# THREAD 1: SAVE A PREFERENCE

thread1 = {"configurable": {"thread_id": str(uuid7())}}
memory_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Hi! Remember: I always wahese to my preferences file."
    )}]},
    config=thread1,
)
nt a window seat, I'm vegetarian, and I have Marriott Bonvoy #1234. "
        "Save tprint("Thread 1 done -- preferences saved.")

Thread 1 done -- preferences saved.


In [100]:
# THREAD 2: BRAND NEW THREAD, BUT THE MEMORY IS STILL THERE

thread2 = {"configurable": {"thread_id": str(uuid7())}}
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What do you know about my preferences?"}]},
    config=thread2,
)
print(result["messages"][-1].content)


Your preferences are:
- You always want a window seat.
- You follow a vegetarian diet.
- You have a Marriott Bonvoy membership with number 1234.


### Key Takeaway

By routing `/memories/` to a `StoreBackend`, we get cross-thread persistence with no special API — the agent just reads and writes files. State (scratch work) and store (long-term memory) coexist via `CompositeBackend`.


### Advanced Memory Patterns: Semantic, Episodic & Procedural

Cognitive scientists distinguish three kinds of long-term memory. The same split is useful for agents:

<table style="font-size: 1.15em">
<tr><th>Type</th><th>What it stores</th><th>Travel example</th></tr>
<tr><td><b>Semantic</b></td><td>Stable facts about entities</td><td>"Alice prefers window seats, vegetarian, Marriott Bonvoy #1234"</td></tr>
<tr><td><b>Episodic</b></td><td>Specific past events</td><td>"Alice's 2024 Paris trip &mdash; loved Le Bristol, complained about A/C"</td></tr>
<tr><td><b>Procedural</b></td><td>Rules / how-to / heuristics</td><td>"Always check baggage fees before comparing flight prices"</td></tr>
</table>

We can give each type its own subdirectory and its own namespace:


In [101]:
def tiered_memory_backend(rt):
    return CompositeBackend(
        default=StateBackend(rt),
        routes={
            "/memories/semantic/":   StoreBackend(rt, namespace=lambda ctx: ("memories", "semantic")),
            "/memories/episodic/":   StoreBackend(rt, namespace=lambda ctx: ("memories", "episodic")),
            "/memories/procedural/": StoreBackend(rt, namespace=lambda ctx: ("memories", "procedural")),
        },
    )

tiered_store = InMemoryStore()

In [104]:
tiered_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a travel concierge with three kinds of long-term memory. Use them deliberately:\n"
        "- `/memories/semantic/preferences.md` -- stable traveler facts (seat, diet, loyalty)\n"
        "- `/memories/episodic/trips/<year>-<city>.md` -- past trip notes\n"
        "- `/memories/procedural/booking_rules.md` -- general booking heuristics\n"
        "Read the relevant files at the start of every conversation."
    ),
    backend=tiered_memory_backend,
    checkpointer=MemorySaver(),
    store=tiered_store,
)

# SEED ALL THREE TIERS WITH SOME USER MEMORIES

tiered_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Remember three things in the right files:\n"
        "1. (semantic) I'm Alice. Window seat, vegetarian, Marriott Bonvoy #1234, no red-eyes.\n"
        "2. (episodic) Last June I went to Paris -- stayed at Hotel Le Bristol, loved the breakfast, complained about the A/C.\n"
        "3. (procedural) Always check baggage fees BEFORE comparing flight prices, and prefer refundable hotel rates within 30 days of travel."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("All three memory tiers seeded.")

All three memory tiers seeded.


In [105]:
# A NEW THREAD CAN READ ALL 3 SAVED MEMORIES

result = tiered_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm planning a trip to Madrid. Pull up everything you remember about me, any past trips, and any booking rules I follow."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print(result["messages"][-1].content)

Here's what I have about you relevant to your trip to Madrid:

Preferences:
- Your name: Alice
- Seat preference: Window seat
- Dietary preference: Vegetarian
- Loyalty program: Marriott Bonvoy #1234
- Flight preference: No red-eye flights

Past Trips to Madrid:
- No past trip notes to Madrid on record.

Booking Rules You Follow:
- Always check baggage fees BEFORE comparing flight prices.
- Prefer refundable hotel rates within 30 days of travel.

Let me know if you want me to assist with anything specific for your Madrid trip!


### Namespace Scoping: Per-User vs Global Memory

If the same agent serves multiple travelers, Alice's preferences must not leak to Bob. The `namespace` callable on `StoreBackend` receives a `BackendContext` whose `runtime.context` includes whatever you pass via `config={"configurable": {...}}`. We'll scope per-user memory by `user_id` and keep a separate `shared` namespace for things that apply to everyone (visa/voltage info, blacklisted suppliers).


In [106]:
from langgraph.config import get_config

def _current_user_id() -> str:
    cfg = get_config() or {}
    return cfg.get("configurable", {}).get("user_id", "anonymous")

def per_user_memory_backend(rt):
    # Important: get_config() is called inside the namespace lambda so that
    # user_id is resolved per store operation (i.e. per request), not frozen
    # at backend construction.
    return CompositeBackend(
        default=StateBackend(rt),
        routes={
            "/memories/user/":   StoreBackend(rt, namespace=lambda ctx: ("user", _current_user_id(), "filesystem")),
            "/memories/shared/": StoreBackend(rt, namespace=lambda ctx: ("shared", "filesystem")),
        },
    )

multiuser_store = InMemoryStore()

multiuser_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a travel concierge serving many travelers. "
        "Save personal preferences to `/memories/user/preferences.md`. "
        "Save universally-applicable rules (visa info, supplier blacklist) to `/memories/shared/rules.md`."
    ),
    backend=per_user_memory_backend,
    checkpointer=MemorySaver(),
    store=multiuser_store,
)

In [107]:
# ALICE WRITES _BOTH_ A PRIVATE _AND_ A SHARED NOTE

alice_cfg = {"configurable": {"thread_id": str(uuid7()), "user_id": "alice"}}
multiuser_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm Alice. Save to my personal preferences: window seat, vegetarian, Marriott Bonvoy #1234. "
        "Also save to shared rules: 'US passport holders need ETIAS for Schengen starting 2025'."
    )}]},
    config=alice_cfg,
)
print("Alice saved both files.")

Alice saved both files.


In [108]:
# BOB (DIFFERENT USER_ID) CHECKS WHAT _HE_ CAN SEE

bob_cfg = {"configurable": {"thread_id": str(uuid7()), "user_id": "bob"}}
result = multiuser_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm Bob. List everything you have under `/memories/user/` AND under `/memories/shared/`."
    )}]},
    config=bob_cfg,
)
print(result["messages"][-1].content)
print("\nExpected: Bob sees the SHARED ETIAS rule but NOT Alice's preferences.")

Under /memories/user/ there are currently no files.

Under /memories/shared/ there is one file:
- rules.md

Expected: Bob sees the SHARED ETIAS rule but NOT Alice's preferences.


### Key Takeaway

The `namespace` callable on `StoreBackend` is your isolation primitive. Tuple components like `("user", user_id, ...)` or `("shared", ...)` cleanly partition the store. Alice's seat preferences stay private; the ETIAS rule is global. The agent sees the same paths regardless — only the underlying namespace differs.


## Part 8: AGENTS.md & Skills

So far we've put the agent's identity inline in `system_prompt`. That works, but it doesn't scale: as your agent grows, its instructions become a sprawling string, and there's no way for it to **learn** by editing them.

Two patterns fix this:

- **`AGENTS.md`** — a markdown file the agent loads at startup as part of its identity. The agent can `edit_file` it during a conversation to record lessons learned. Pass via `memory=["/AGENTS.md"]`.
- **Skills (`SKILL.md`)** — small markdown files describing *how to do a specific thing* (write an itinerary in our standard format, produce a budget summary table, etc.). The agent only loads a skill when it needs it (progressive disclosure). Pass via `skills=["/skills/"]`.

Let's define an `AGENTS.md` for our travel planner and four skills.


In [109]:
from deepagents.backends.utils import create_file_data

AGENTS_MD = """# Travel Planner

You are an expert travel concierge. You research, quote, and produce polished trip plans.

## Workflow

1. **Understand**   -- ask 2-3 clarifying questions if the request is vague (dates, budget, style, party size)
2. **Load skills**  -- for every deliverable the user asks for (itinerary, budget summary, packing list, travel brief), `read_file` the matching `/skills/<name>/SKILL.md` BEFORE step 3. This is mandatory, even if the format feels obvious -- the SKILL.md is the only authoritative source for our exact format
3. **Plan**         -- use `write_todos` to outline the work
4. **Delegate**     -- use `task()` to call the hotel-search, flight-search, and activity-search subagents IN PARALLEL where possible
5. **Synthesize**   -- combine subagent reports into `/itinerary.md` and `/budget.md` in one pass; do NOT pause for confirmation between research and synthesis
6. **Remember**     -- save lasting traveler facts to `/memories/user/preferences.md`

## Rules

- After step 1's clarifying questions, do not ask for any further user confirmation. Make a reasonable choice from the subagents' options (cheapest, best-rated, or best fit for stated style) and proceed straight through to writing the deliverable file

- Booking tools (book_flight, book_hotel) require explicit user approval
- Cite hotel/flight IDs from quotes so the user can ask you to book them
- When referencing file paths, use backtick formatting like `path/file.md`
"""

### Now let's define our agent's Skills

Each skill lives at `/skills/<name>/SKILL.md`. The agent will discover them via `glob`/`ls` and read the relevant one when the task calls for it.


In [110]:
ITINERARY_SKILL = """---
name: itinerary-format
description: Produce a day-by-day travel itinerary using our standard structure.
---

# Itinerary Format

Use this exact structure when writing `/itinerary.md`:

```
# <Destination> -- <Trip Dates>

## Trip Overview
- Travelers: ...
- Style: ...
- Budget: ...

## Day 1 -- <Date> (<Day of week>)
- Morning:   ...
- Afternoon: ...
- Evening:   ...
- Lodging:   ...

## Day 2 -- <Date> ...
(repeat per day)

## Logistics
- Flights:    ...
- Transit:    ...
- Currency:   ...
```

Always include time blocks (Morning/Afternoon/Evening). Always end with a Logistics section.
"""

BUDGET_SKILL = """---
name: budget-summary
description: Produce a cost-breakdown table for a trip.
---

# Budget Summary Format

Use this exact structure when writing `/budget.md`:

```
# Budget -- <Destination>, <Dates>

| Category    | Item                  | Cost    |
|-------------|-----------------------|---------|
| Flights     | <airline + route>     | $...    |
| Lodging     | <hotel x N nights>    | $...    |
| Activities  | <tour name>           | $...    |
| Food        | <est. per day x days> | $...    |
| Buffer 10%  |                       | $...    |
| **TOTAL**   |                       | **$...**|
```

Always include a 10% buffer line. Total in bold.
"""

PACKING_SKILL = """---
name: packing-list
description: Produce a climate- and activity-aware packing checklist.
---

# Packing List Format

Use this structure when writing `/packing_list.md`:

```
# Packing List -- <Destination>, <Dates>

## Documents
- [ ] Passport (+ photocopy)
- [ ] Visa / ETIAS confirmation if applicable
- [ ] Travel insurance card

## Climate-appropriate clothing
- [ ] (specific items based on forecast)

## Activity-specific gear
- [ ] (e.g. hiking shoes if hiking, swim gear if beach)

## Tech
- [ ] Plug adapter (specify type)
- [ ] Chargers
```
"""

BRIEF_SKILL = """---
name: travel-brief
description: Produce a one-page traveler brief with logistics essentials.
---

# Travel Brief Format

Use this structure when writing `/travel_brief.md`:

```
# Travel Brief -- <Destination>

- **Currency:**    <code> (~<rate vs USD>)
- **Plug type:**   <letter> (<voltage>)
- **Visa:**        <required? on arrival? ETIAS?>
- **Emergency:**   <local 911-equivalent>
- **Time zone:**   <UTC offset>
- **Tipping:**     <norms>

## Key phrases
- Hello: ...
- Thank you: ...
- Where is the bathroom?: ...
```
"""

### Put It All Together (AGENTS.md + Skills)

We seed the virtual filesystem with the AGENTS.md and the four skill files using `create_file_data`, then point the agent at them via `memory=` and `skills=`.


In [111]:
seed_files = {
    "/AGENTS.md": create_file_data(AGENTS_MD),
    "/skills/itinerary-format/SKILL.md": create_file_data(ITINERARY_SKILL),
    "/skills/budget-summary/SKILL.md":   create_file_data(BUDGET_SKILL),
    "/skills/packing-list/SKILL.md":     create_file_data(PACKING_SKILL),
    "/skills/travel-brief/SKILL.md":     create_file_data(BRIEF_SKILL),
}

# Track skill reads from anywhere (orchestrator OR subagent), since the
# orchestrator's `result["messages"]` doesn't include subagent tool calls.
tracked_skill_reads: list[str] = []

@wrap_tool_call
def track_skill_reads(request, handler):
    if request.tool_call["name"] == "read_file":
        path = request.tool_call.get("args", {}).get("file_path", "")
        if "skills/" in path and path.endswith("SKILL.md"):
            tracked_skill_reads.append(path)
    return handler(request)

skill_agent = create_deep_agent(
    model=model,
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    middleware=[log_tool_calls, track_skill_reads],
    checkpointer=MemorySaver(),
)
def show_file(result, path):
    file = result.get("files", {}).get(path)
    if not file:
        print(f"(no {path} found)")
        return
    print(f"\n----- {path} -----")
    print("".join(file["content"]))


In [113]:
# TEST 1: ASK FOR AN ITINERARY -- THE AGENT SHOULD LOAD THE ITINERARY-FORMAT SKILL

tracked_skill_reads.clear()

result = skill_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Before producing any output, call `read_file` on `/skills/itinerary-format/SKILL.md` to load the exact format. "
        "Then plan a 3-day Porto trip from SFO, October 14-16, solo traveler, "
        "mid-budget (~$1500 total), food and architecture focused, "
        "and produce the itinerary in `/itinerary.md` following the loaded skill exactly."
    )}], "files": seed_files},
    config={"configurable": {"thread_id": str(uuid7())}},
)

print(f"\nSkills loaded during run: {tracked_skill_reads or '(none)'}\n")
print(result["messages"][-1].content)

show_file(result, "/itinerary.md")


➤  read_file(file_path=/skills/itinerary-format/SKILL.md, offset=0, limit=100)
✓  read_file done
➤  write_todos(todos=[{'content': 'Search for flight options )
✓  write_todos done
➤  task -> activity-search: Find food and architecture focused activities and tours in Porto on October 14, ...
➤  task -> hotel-search: Find mid-budget lodging options in Porto from October 14 to October 16 suitable ...
➤  task -> flight-search: Find flights from San Francisco (SFO) to Porto (OPO) for departure on October 14...
✓  task done
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Search for flight options )
✓  write_todos done
➤  write_file(file_path=/itinerary.md, content=# Porto -- October 14-16

## Trip Overvi)
✓  write_file done

Skills loaded during run: ['/skills/itinerary-format/SKILL.md']

I have created your 3-day Porto itinerary from October 14-16 as a solo traveler with a mid-budget focus on food and architecture. It includes flights, lodging, and daily activities following th

In [114]:
# TEST 2: ASK FOR A BUDGET SUMMARY -- THE AGENT SHOULD LOAD THE BUDGET-SUMMARY SKILL ON DEMAND

tracked_skill_reads.clear()

result = skill_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Before producing any output, call `read_file` on `/skills/budget-summary/SKILL.md` to load the exact format. "
        "Then plan a 4-day Marrakech trip from SFO, October 22-25, two travelers, "
        "mid-budget (~$2500 total), cultural and food focused, "
        "and produce a budget summary in `/budget.md` following the loaded skill exactly."
    )}], "files": seed_files},
    config={"configurable": {"thread_id": str(uuid7())}},
)

print(f"\nSkills loaded during run: {tracked_skill_reads or '(none)'}\n")
print(result["messages"][-1].content)
show_file(result, "/budget.md")



➤  read_file(file_path=/skills/budget-summary/SKILL.md)
✓  read_file done
➤  task -> flight-search: Research mid-budget flights from San Francisco (SFO) to Marrakech (RAK) for Octo...
✓  task done
➤  task -> hotel-search: Research mid-budget lodging in Marrakech for October 22-25 (3 nights) for 2 trav...
✓  task done
➤  task -> activity-search: Research cultural and food-focused activities in Marrakech for October 22-25 for...
✓  task done
➤  write_file(file_path=/budget.md, content=# Budget -- Marrakech, October 22-25

| )
✓  write_file done

Skills loaded during run: ['/skills/budget-summary/SKILL.md']

I have produced the budget summary for your Marrakech trip in the file `/budget.md`. The total comes to $3358 with the selected flights, lodging, activities, and food estimates including a 10% buffer. This slightly exceeds your $2500 target due to flights. If you want, I can help adjust lodging or food budget to better fit $2500 total. Let me know how you would like to proceed.

-----

### Key Takeaway

`AGENTS.md` is your agent's persistent identity — it can be edited (by you, or by the agent itself) and survives across runs. **Skills** are bite-sized, on-demand instructions for specific output formats. Together they replace ever-growing system prompts with a clean, file-based knowledge layer.


## Part 9: The Complete Travel Planner

Time to assemble everything we've built:

- 3 specialist subagents (hotel / flight / activity)
- Custom `search_travel` + 3 mock pricing tools
- 2 booking tools gated by HITL
- `AGENTS.md` for identity + 4 skills for output formats
- Tool-call logging middleware
- `CompositeBackend` routing per-user `/memories/` to a `StoreBackend`
- `MemorySaver` checkpointer

Then run a real trip planning task end-to-end.


In [115]:
final_store = InMemoryStore()
final_checkpointer = MemorySaver()

travel_planner = create_deep_agent(
    model=model,
    tools=[book_flight, book_hotel],   # BOOKINGS LIVE ON THE ORCHESTRATOR (GATED BY HUMAN-IN-THE-LOOP CONTROLS)
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    middleware=[log_tool_calls],
    backend=per_user_memory_backend,
    checkpointer=final_checkpointer,
    store=final_store,
    interrupt_on={"book_flight": True, "book_hotel": True},
)


In [116]:
# RUN AN END-TO-END TRIP PLANNING REQUEST

trip_thread = {"configurable": {"thread_id": str(uuid7()), "user_id": "alice"}}

result = travel_planner.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm Alice. Plan 7 days in Lisbon for two travelers, departing from SFO in late September. "
        "Budget around $3,000 for flights + lodging combined. "
        "We're foodies and we'd like a couple of light hikes near the city; "
        "book a centrally located mid-range boutique hotel. "
        "Make reasonable choices on flights and lodging without asking for confirmation. "
        "Before producing output, call `read_file` on `/skills/itinerary-format/SKILL.md` and "
        "`/skills/budget-summary/SKILL.md`, then produce both deliverables in our standard formats. "
        "Save my preferences (vegetarian, window seat) for next time."
    )}], "files": seed_files},
    config=trip_thread,
)

print("\n--- ALL MESSAGES ---")
for msg in result["messages"]:
    msg.pretty_print()


print("\n--- FINAL RESPONSE ---")
print(result["messages"][-1].content)


➤  read_file(file_path=/skills/itinerary-format/SKILL.md)
➤  read_file(file_path=/skills/budget-summary/SKILL.md)
✓  read_file done
✓  read_file done
➤  write_todos(todos=[{'content': 'Search for reasonable flig)
✓  write_todos done
➤  task -> flight-search: Find reasonable flight options from San Francisco International Airport (SFO) to...
➤  task -> activity-search: Research and find 2 food-related activities/experiences and 2 light hikes near L...
➤  task -> hotel-search: Search for centrally located mid-range boutique hotels in Lisbon for 7 nights in...
✓  task done
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Search for reasonable flig)
✓  write_todos done
➤  edit_file(file_path=/memories/user/preferences.md, old_string=, new_string=traveler: Alice
preferences:
  seating: )
✓  edit_file done
➤  write_todos(todos=[{'content': "Create a new user preferen)
✓  write_todos done
➤  write_file(file_path=/memories/user/preferences.md, content=traveler: Alice
preferences:
 

In [66]:
# INSPECT WHAT THE PLANNER PRODUCED

state = travel_planner.get_state(trip_thread)
files = state.values.get("files", {})
for path in sorted(files.keys()):
    if path.startswith(("/AGENTS", "/skills/")):
        continue   # skip the seeded files
    print(f"\n=== {path} ===")
    print("\n".join(files[path]["content"]))


### Key Takeaway

A production-style Deep Agent is just composition: a base harness (`create_deep_agent`) plus the layers we built one at a time — subagents, custom tools, memory backends, HITL gates, identity files, skills, observability middleware. Each layer is small and independently testable. The travel domain happens to be a great fit, but the same recipe works for any orchestrator-plus-specialists workload.


## Part 10: Deploying Your Travel Planner

Everything we've built so far runs in this notebook. To put it in front of real travelers you need a hosted endpoint -- something a web app, Slack bot, or scheduled job can call.

Deep Agents ships a CLI that turns an agent directory into a deployed service on **LangGraph Platform**. The deployed agent gets, for free:

- HTTP + SDK clients (Python / TypeScript)
- Server-side **checkpointer + store** (real persistence, not the `InMemoryStore` we've been using)
- HITL endpoints that match the `interrupt_on` config
- Built-in tracing in **LangSmith**
- Optional Agent Protocol / MCP / A2A surfaces

A deployable agent is just a directory with three or four files.


### Project Structure

A typical layout for a deployable travel planner looks like this:

```
travel-planner/
├── deepagents.toml      # deploy config: name, model, sandbox settings
├── AGENTS.md            # agent identity (what we built in Part 8)
├── skills/              # on-demand skills (what we built in Part 8)
│   ├── itinerary-format/SKILL.md
│   ├── budget-summary/SKILL.md
│   ├── packing-list/SKILL.md
│   └── travel-brief/SKILL.md
├── tools.py             # @tool definitions (search_travel + mock pricing)
├── subagents.py         # the three SubAgent dicts
└── agent.py             # create_deep_agent(...) wired up like Part 9
```

The `agents/deep_agent/` directory in this repo is an example of the same shape (for the original research agent).


In [ ]:
# Inspect the existing example layout in this repo
import os

example_dir = os.path.join(os.path.dirname(os.getcwd()), "agents", "deep_agent")
for root, dirs, files in os.walk(example_dir):
    rel = root.replace(example_dir, "") or "."
    depth = rel.count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or 'deep_agent'}/")
    for f in sorted(files):
        print(f"{indent}  {f}")


### `deepagents.toml` -- the Deploy Config

The deploy config is small. The minimum is a name and model; from there you can tune sandbox settings, dependencies, and which graph to expose.


In [ ]:
# Example deepagents.toml for the travel planner we built in this notebook.
# (We're showing it as a string here; in a real project this is a file.)

TRAVEL_PLANNER_TOML = '''
[agent]
name = "travel-planner"
model = "openai:gpt-4.1-mini"
graph = "agent.py:agent"

[deps]
packages = [
    "deepagents>=0.4.12",
    "tavily-python",
]

[env]
required = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
'''
print(TRAVEL_PLANNER_TOML)


### CLI Commands

The `deepagents` CLI gives you three commands you'll actually use:

```bash
# Scaffold a brand-new agent project (creates deepagents.toml, AGENTS.md, skills/)
deepagents init travel-planner

# Run locally for development -- gives you a /threads endpoint on http://localhost:2024
deepagents dev --config deepagents.toml --port 2024

# Deploy to LangGraph Platform / LangSmith
deepagents deploy --config deepagents.toml
```

`dev` is the iteration loop -- you can hit the local server with the LangGraph SDK, point a chat UI at it, and use the LangSmith tracer to inspect every tool call. `deploy` ships it to a hosted environment with the same API.


### Dry Run -- See What Would Be Deployed

`--dry-run` prints the deployment plan (which files get bundled, which env vars are required, target environment) without actually deploying. Always do this first.

```bash
deepagents deploy --config deepagents.toml --dry-run
```

The cell below would invoke that as a subprocess. It's commented out because the `deepagents` CLI isn't installed in this notebook's venv -- install it with `pip install "deepagents[cli]"` first.


In [ ]:
# Uncomment after `pip install "deepagents[cli]"`:
#
# import subprocess
# result = subprocess.run(
#     ["deepagents", "deploy", "--config", "deepagents.toml", "--dry-run"],
#     cwd="path/to/travel-planner",
#     capture_output=True, text=True,
# )
# print(result.stdout or result.stderr)


### Persistence Changes When You Deploy

The biggest practical difference between notebook code and deployed code is **persistence**. In the notebook we used:

- `MemorySaver()` -- in-process checkpointer
- `InMemoryStore()` -- in-process store

Both vanish when the Python process exits. On LangGraph Platform you get a **managed Postgres-backed checkpointer and store automatically** -- you remove those two arguments from `create_deep_agent` (or set them to `None`) and the platform injects the real ones.

That's why the per-user namespace work in Part 7 matters: as soon as you deploy, multiple travelers will share the same store, and the `("user", user_id, ...)` namespace is what keeps Alice's window-seat preference from leaking to Bob.


### Deploy Recap

| File | Purpose |
| --- | --- |
| `deepagents.toml` | Deploy config: agent name, model, dependencies, env vars |
| `AGENTS.md` | Agent identity (loaded via `memory=` at runtime) |
| `skills/<name>/SKILL.md` | On-demand capabilities (loaded via `skills=` at runtime) |
| `agent.py` | Where you call `create_deep_agent(...)` and export the graph |
| `mcp.json` (optional) | MCP server configuration if your agent uses MCP tools |

| Command | What it does |
| --- | --- |
| `deepagents init <name>` | Scaffold a new agent project |
| `deepagents dev` | Run locally with hot-reload + tracing |
| `deepagents deploy --dry-run` | Preview the deployment plan |
| `deepagents deploy` | Ship to LangGraph Platform |


## Part 11: Next Steps

Where to take this travel planner:

- **Real APIs** -- swap the mock tools for Amadeus / Skyscanner / Booking.com / Viator
- **Calendar integration** -- have the agent write `/itinerary.ics` (iCal) and email it
- **Multi-leg trips** -- one orchestrator coordinating multiple per-leg subagents
- **Group travel** -- merge preferences from multiple `user_id`s when planning for a party
- **Budget enforcement** -- a custom middleware that rejects or warns when subagent quotes blow the budget envelope
- **Episodic memory at scale** -- after each trip, have a "trip post-mortem" subagent that writes `/memories/episodic/trips/<year>-<city>.md` with what worked and what didn't, so the next plan benefits from history

Where to take Deep Agents in general:

- Build your own middleware for guardrails, cost tracking, PII redaction
- Plug in `LangSmithSandbox` or `FilesystemBackend` to give the agent real shell `execute` capability
- Use `CompiledSubAgent` to wrap an existing LangGraph agent as a subagent of a Deep Agent

Read more: <https://github.com/langchain-ai/deepagents>
